In [ ]:
pip install flask flask-cors

In [ ]:
pip install requests

In [ ]:
pip install langchain langchain_community langchain-ollama

In [4]:
# Load Data from SQLite and Create Documents
import sqlite3
from langchain.schema import Document

sqlite_file = "./courses.sqlite"
table_name = "merged_courses"
content_columns = [
    "title", "instructor", "learning_obj", "course_contents", "prerequisites",
    "readings", "applicability", "workload", "credits", "evaluation", 
    "time", "duration", "course_type"
]

def load_db(sqlite_file, table_name, content_columns):
    conn = sqlite3.connect(sqlite_file)
    cursor = conn.cursor()
    query = f"SELECT * FROM {table_name}"
    cursor.execute(query)
    rows = cursor.fetchall()
    column_names = [desc[0] for desc in cursor.description]
    conn.close()

    documents = []
    for row in rows:
        meta_dict = {col: row[column_names.index(col)] for col in column_names}

        title = str(meta_dict.get("title", ""))
        instructor = str(meta_dict.get("instructor", ""))
        learning_obj = str(meta_dict.get("learning_obj", ""))
        course_contents = str(meta_dict.get("course_contents", ""))
        prerequisites = str(meta_dict.get("prerequisites", ""))
        readings = str(meta_dict.get("readings", ""))
        applicability = str(meta_dict.get("applicability", ""))
        workload = str(meta_dict.get("workload", ""))
        credits = str(meta_dict.get("credits", ""))
        evaluation = str(meta_dict.get("evaluation", ""))
        time = str(meta_dict.get("time", ""))
        duration = str(meta_dict.get("duration", ""))
        course_type = str(meta_dict.get("course_type", ""))

        structured_text = f"""
Title: {title}
Instructor: {instructor}
Learning Objectives: {learning_obj}
Course Contents: {course_contents}
Prerequisites: {prerequisites}
Readings: {readings}
Applicability: {applicability}
Workload: {workload}
Credits: {credits}
Evaluation: {evaluation}
Time: {time}
Duration: {duration}
Course Type: {course_type}
""".strip()

        # Storing as page_content
        doc = Document(page_content=structured_text, metadata=meta_dict)
        documents.append(doc)

    return documents


documents = load_db(sqlite_file, table_name, content_columns)

# Inspection of documents
for i, doc in enumerate(documents):
    print(f"\n=== Document #{i} ===")
    print(doc.page_content)
    print("--- End of Document ---")

# print(documents[:2])

/Users/jonayedhossenantor/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



=== Document #0 ===
Title: Automotive Technology Management
Instructor: Prof Dr Michael Dornieden
examples of automotive industrymanagement tools to real-worldr OEM´srojectskers
5. Innovation managementrocessurer, Original Equipment Supplier and
Prerequisites: There are no formal requirements.
Readings: Nieuwenhuis, P. / Wells, P.: Global Automotive Industry, John Wiley & Sons, 2015\nMunson, C.: The Supply Chain Management Casebook, FT Press, 2013Diehlmann, J. / Häcker, J.: Automotive Management, 2nd ed., Oldenbourg Verlag, 2013 Myerson, P. A.: Lean and Technology: Working Hand in Hand to Enable and Energize Your Global Supply Chain, Pearson Education, 2017F urther references will be given during the classes.
Applicability: This course is in particular applicable to the following Master programmes: International Business and Economics (M.A.; “IBE”).
This course is also applicable to other business-oriented Master programmes offered by Schmalkalden University of Applied Sciences.
▪ Rea

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000, chunk_overlap=400, add_start_index=True
)
all_splits = text_splitter.split_documents(documents)
print(all_splits)

[Document(metadata={'rowid': 1, 'file_loc': 'E:\\HSM\\project_hmi\\webappcoursespot\\courses\\zqm_modul\\en\\Automotive Technology Mgmt_MD.pdf', 'title': 'Automotive Technology Management', 'instructor': 'Prof Dr Michael Dornieden', 'learning_obj': 'Identify relevant market framework conditions of carmakers\r▪ Realize the economic importance of the automotive industry\r▪ Know automotive key figures in respect to particular countries\r▪ Categorize the product portfolio of automotive suppliers\r▪ Examine and understand future trends of automotive markets and\rresource requirements of carmakers (e.g. electromobility)\r▪ Critically evaluate strategic alliances between carmakers currently in\rpractical company use and develop an understanding of how they can\rbe improved in order to reach the desired organizational goals\r▪ Know theoretical foundations of product lifecycle management\r▪ Characterize the six phases of the Generic Product Development\rProcess and demonstrate its application t

In [ ]:
pip install langchain-chroma

In [7]:
# Persist the vector store
import os
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores.chroma import Chroma
import logging

VECTORSTORE_DIR = "./chroma_vectorDB"
def initialize_vectorstore():
    if not os.path.exists(VECTORSTORE_DIR):
        local_embeddings = OllamaEmbeddings(model="all-minilm")
        vectorstore = Chroma.from_documents(
            documents=all_splits,
            embedding=local_embeddings,
            persist_directory=VECTORSTORE_DIR
        )
        vectorstore.persist()
        logging.info("Vector store created and persisted to disk.")
    else:
        logging.info("Vector store already exists. Skipping embedding computation.")


In [8]:
initialize_vectorstore()

In [9]:
# Load the vector store for querying
def load_vectorstore():
    local_embeddings = OllamaEmbeddings(model="all-minilm")
    return Chroma(
        persist_directory=VECTORSTORE_DIR,
        embedding_function=local_embeddings
    )

vectorstore = load_vectorstore()

# print(vectorstore)

/var/folders/g2/g3tj8_9x6cv7gfxf821v0scr0000gn/T/ipykernel_1278/813056856.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  local_embeddings = OllamaEmbeddings(model="all-minilm")
/var/folders/g2/g3tj8_9x6cv7gfxf821v0scr0000gn/T/ipykernel_1278/813056856.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  return Chroma(


In [10]:
from langchain_ollama.llms import OllamaLLM

In [11]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 20})

In [12]:
docs = retriever.get_relevant_documents("Can you tell me something about Prof. Dr. Englmeier?")
for doc in docs:
    print(doc.page_content)

/var/folders/g2/g3tj8_9x6cv7gfxf821v0scr0000gn/T/ipykernel_1278/356835207.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents("Can you tell me something about Prof. Dr. Englmeier?")


Title: Strategic International Human Resources Management
Instructor: Prof. Dr. Andreas Kammel
Strategic International Human Resources Managementto compute, analyze, interpret, apply, synthesize, and evaluate the complex knowledge of
2.9 HRM Trends and the Changing Workplaced Global Institutional Contextational setting.ctices in organizationsization designely:
Prerequisites: There are no formal requirements.
Dowling, P.J./Festing, M./Engle, A.D. International Human Resources Management, Hampshire UK
Applied Sciences.so applicable to other business-oriented Master programmes offered by Schmalkalden University ofonal Business and
- exam preparationacademic research project)
Credits: 6 
- student presentation in class (20%)xamination, 90 minutes (80%)
Time: first academic year
Duration: 1 semester
Course Type: Elective
Title: Economic Philosophy
Instructor: Prof Dr Robert Richert
▪ Design an economic and social order that fits to the relevant cultureon, culture and
8. Design of a promisin

In [13]:
llm = OllamaLLM(model="llama3.1")

In [14]:
# RAG
class RAGApplication:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm

    def run(self, question: str):
        logging.info(f"User query: {question}")

        # Retrieve top matches from the vector store
        relevant_docs = self.retriever.get_relevant_documents(question)

        meta_lines = []
        for doc in relevant_docs:
            title = doc.metadata.get("title", "Unknown Title")
            instructor = doc.metadata.get("instructor", "")
            applicability = doc.metadata.get("applicability", "")
            short_descr = doc.metadata.get("course_contents", "")[:200]
            meta_lines.append(
                f"**Official Course Title**: {title}\n"
                f"Instructor: {instructor}\n"
                f"Applicability: {applicability}\n"
                f"Description (Brief): {short_descr}\n"
            )

        context = "\n\n".join(meta_lines)

        print("context: ", context)

        prompt = f"""
        You are an AI assistant with access to a knowledge base containing course information. 
        Your job is to answer the user’s question as accurately as possible using ONLY the provided context.
        If the answer is not explicitly stated in the context, explain the reason in disclaimer.
        {context}

        The user asked: "{question}"

        Instructions:
        1. You have access to context (excerpts from the database). 
        2. If the user queries about a specific instructor, list each course taught by that instructor.
        3. For each such course, display a short description (e.g., from 'course_contents' or 'learning_obj').
        4. If the user wants to know which department or Master’s program is relevant, look at 'Applicability:' in each course's context.
        5. Return the answer using HTML tags such as h1, h2, h3, h4, h5, h6, p, ul, li etc.
        6. Use only the provided context to answer the question.
        7. Do NOT use Markdown pipe syntax. Only output raw HTML <table> tags.
        8. You can use HTML table if there is any comparison.
        9. If you do not see any mention of a specific course or detail in the context, say:
        "<p><strong>Disclaimer:</strong> I'm not sure based on the provided context.</p>"

        Answer:
        """

        response = self.llm.invoke(prompt)

        return {"response": response, "context": context}


In [ ]:
pip install --upgrade flask-cors

In [16]:
from flask import Flask, request, jsonify
from flask_cors import CORS, cross_origin

app = Flask(__name__)
CORS(app, support_credentials=True)

rag_app = RAGApplication(retriever, llm)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

@app.route("/ask", methods=["POST"])
@cross_origin(supports_credentials=True)
def ask():
    try:
        data = request.json
        if not data or "query" not in data:
            return jsonify({"error": "No query provided"}), 400

        question = data["query"].strip()
        if not question:
            return jsonify({"error": "Query cannot be empty"}), 400

        # Using RAG application
        result = rag_app.run(question)
        logging.info(f"Question: {question} | Answer: {result}")

        return jsonify({
            "response": result["response"],
            "context": result["context"]
        })

    except Exception as e:
        logging.error(f"Error occurred: {str(e)}", exc_info=True)
        return jsonify({"error": "Internal error"}), 500

if __name__ == "__main__":
    app.run(debug=True, host='0.0.0.0', port=5050, use_reloader=False)



 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5050
 * Running on http://10.55.0.42:5050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [28/Jan/2025 21:48:25] "OPTIONS /ask HTTP/1.1" 200 -


context:  **Official Course Title**: Industrial Economics
Instructor: Kai Hüschelrath
Applicability: This course is in particular applicable to the following Master programmes: International Business and Economics (M.A.; “IBE”), Finance (M.Sc.)
This course is also applicable to other business-oriented Master programmes offered by Schmalkalden University of Applied Sciences.
Paice Discriminationnceodelsction


**Official Course Title**: Regional Economics
Instructor: Prof Dr Wiebke Störmann
Sciences.se is also applicable to other business-oriented Master programmes offered by Schmalkalden University of Appliednomics (M.A.;
The Bid-Rent Model for a FirmUrban Economygeout


**Official Course Title**: Economic Philosophy
Instructor: Prof Dr Robert Richert
programmes offered by Schmalkalden University of Applied Sciences.).ing Master
1.3 Transparency International’s corruption perceptiohy, religion, culture, economic success


**Official Course Title**: Purchasing Strategy
Instructor: Prof 

INFO:werkzeug:127.0.0.1 - - [28/Jan/2025 21:51:05] "POST /ask HTTP/1.1" 200 -
